# Byte Pair Encoding (BPE) helper functions

In [2]:
word_freqs = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
}

# inital stage of getting every pair count in my dictonary 
def init_vocab(word_freqs):
    pair_counter = {}
    for key, value in word_freqs.items():
        for i in range(len(key) -1):
            if (key[i], key[i+1]) not in pair_counter:
                pair_counter[(key[i], key[i+1])] = value
            else:
                pair_counter[(key[i], key[i+1])] += value
    return pair_counter

counter = init_vocab(word_freqs)
print(counter)

# finidng which pair has the highest count
def most_frequent_pair(pair_counter):
    highest_key = max(pair_counter, key = pair_counter.get)
    return highest_key

most_frequent = most_frequent_pair(counter)
print(most_frequent)

# converting string representation using list
def word_to_symbols(word):
    return list(word)

# merging based on the pair with the highest count
def merge_pair(symbols, pair):
    new_temp = []
    i = 0
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i+1] == pair[1]:
            new_temp.append(symbols[i] + symbols[i+1])
            i += 2   # skip both consumed symbols
        else:
            new_temp.append(symbols[i])
            i += 1   # move ahead by 1, this symbol wasn't part of a merge
    return new_temp

test = merge_pair(['n','e','w','e','s','t'], ('e','s'))
print(test)


word_symbols = {}
for word in word_freqs:
    word_symbols[word] = word_to_symbols(word)

print(word_symbols)

#merging the highest pair to the vocab in list type, word_symbols is a dictionary holding this result for every word
def apply_merge_to_vocab(word_symbols, pair):
    updated = {}
    for word, lst in word_symbols.items():
        updated[word] = merge_pair(lst, pair)
    return updated

updated = apply_merge_to_vocab(word_symbols, ('e', 's'))
print(updated)

# recounting pairs from updated structure(merged pair in each word)
def count_pairs(word_symbols, word_freqs):
    pair_counter ={}
    for word, symbols in word_symbols.items():
        freq = word_freqs[word]
        for i in range((len(symbols) -1)):
            pair = (symbols[i], symbols[i+1])
            if pair not in pair_counter:
                pair_counter[pair] = freq
            else:
                pair_counter[pair] += freq
    return pair_counter

updated_counter = count_pairs(updated, word_freqs)
print(updated_counter)

{('l', 'o'): 7, ('o', 'w'): 7, ('w', 'e'): 8, ('e', 'r'): 2, ('n', 'e'): 6, ('e', 'w'): 6, ('e', 's'): 9, ('s', 't'): 9, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'e'): 3}
('e', 's')
['n', 'e', 'w', 'es', 't']
{'low': ['l', 'o', 'w'], 'lower': ['l', 'o', 'w', 'e', 'r'], 'newest': ['n', 'e', 'w', 'e', 's', 't'], 'widest': ['w', 'i', 'd', 'e', 's', 't']}
{'low': ['l', 'o', 'w'], 'lower': ['l', 'o', 'w', 'e', 'r'], 'newest': ['n', 'e', 'w', 'es', 't'], 'widest': ['w', 'i', 'd', 'es', 't']}
{('l', 'o'): 7, ('o', 'w'): 7, ('w', 'e'): 2, ('e', 'r'): 2, ('n', 'e'): 6, ('e', 'w'): 6, ('w', 'es'): 6, ('es', 't'): 9, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'es'): 3}


# Byte Pair Encoding (BPE) Algorithm

In [5]:
def train_bpe(word_freqs, num_merges):
    first_counter = init_vocab(word_freqs)
    word_symbols = {}
    for word in word_freqs:
        word_symbols[word] = word_to_symbols(word)

    store_pair = []
    for iteration in range(num_merges):
        pair = most_frequent_pair(first_counter)
        store_pair.append(pair)
        updated_symbols = apply_merge_to_vocab(word_symbols, pair)
        word_symbols = updated_symbols
        updated_counter = count_pairs(updated_symbols, word_freqs)
        first_counter = updated_counter

    return store_pair, word_symbols

store_pair, word_symbols = train_bpe(word_freqs,3)
print(store_pair)
print(word_symbols)

def tokenize(word, merges):
    symbols = word_to_symbols(word)
    for pair in merges:
        symbols = merge_pair(symbols, pair)
    return symbols

new = tokenize('lwrsesrwes', store_pair)
print(new)

[('e', 's'), ('es', 't'), ('l', 'o')]
{'low': ['lo', 'w'], 'lower': ['lo', 'w', 'e', 'r'], 'newest': ['n', 'e', 'w', 'est'], 'widest': ['w', 'i', 'd', 'est']}
['l', 'w', 'r', 's', 'es', 'r', 'w', 'es']
